<a href="https://colab.research.google.com/github/Matheusbcy/22-machine-learning-projects/blob/main/Basico/Tel%20Churn/Telco_Customer_Churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import warnings
warnings.filterwarnings('ignore')

# Telco Customer Chrun

## **Definindo o problema**

1. **Qual a variavel alvo (target)**  
**Resposta**: Churn (saber se o cliente vai ou não cancelar o serviço)
2. **Qual tipo de classificação?**  
**Resposta** Classificação Binaria
3. **Qual contexto do negocio?**  
**Resposta**: Prever quais clientes têm maior chance de cancelar (Churn) para que a empresa possa agir antes oferecendo promoções, suporte ou melhorias.

## **Problemas de negocio envolvidos**

* Customer Retention (retenção de clientes)
* Customer Lifetime Value (CLV)
* Marketing Estratégico
* Redução do Custo de Aquisição de Cliente (CAC)

## Coleta de dados

1. **Identificar fonte de dados**  
Dados obtidos no [Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn?resource=download)
2. **Verificar disponibilidade e qualidade**  
Sem dados nulos ✅  

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

In [4]:
df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.drop(["customerID"], axis = 1, inplace = True)
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Pré-processamento dos dados

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [6]:
col_yes_no = ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]

for col in col_yes_no:
  df[col] = df[col].map({"Yes": 1, "No": 0})

In [7]:
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

In [8]:
np.unique(df["MultipleLines"])

array(['No', 'No phone service', 'Yes'], dtype=object)

In [9]:
np.unique(df["InternetService"])

array(['DSL', 'Fiber optic', 'No'], dtype=object)

In [10]:
np.unique(df["Contract"])

array(['Month-to-month', 'One year', 'Two year'], dtype=object)

In [11]:
np.unique(df["PaymentMethod"])

array(['Bank transfer (automatic)', 'Credit card (automatic)',
       'Electronic check', 'Mailed check'], dtype=object)

In [12]:
X = df.drop(["Churn"], axis = 1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

preprocess = ColumnTransformer(
    transformers = [
        ("cat", OneHotEncoder(handle_unknown= "ignore"), cat_cols)
    ], remainder='passthrough'
)

## Tunning de parâmetros - pipeline

## DecisionTreeClassifier

In [ ]:
pipe = Pipeline(steps = [
    ("preprocess", preprocess),
    ("classifier", DecisionTreeClassifier(random_state = 42))
])

params = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__splitter": ["best", "random"],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 5, 10]
}

grid_search = GridSearchCV(estimator=pipe, param_grid = params)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_
best_result = grid_search.best_score_

print(best_params)
print(f"Melhor score: {best_result:.4f}")

## RandomForestClassifier

In [ ]:
pipe = Pipeline(steps = [
    ("preprocess", preprocess),
    ("classifier", RandomForestClassifier(random_state = 42))
])

params = {
    "classifier__n_estimators": [10, 40, 100, 150],
    "classifier__criterion": ["gini", "entropy"],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1,  5, 10]}

grid_search = GridSearchCV(estimator=pipe, param_grid = params)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_
best_result = grid_search.best_score_

print(best_params)
print(f"Melhor score: {best_result:.4f}")

## kNN

In [ ]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocess_knn = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ],
    remainder='passthrough'
)

pipe = Pipeline(steps = [
    ("preprocess", preprocess_knn),
    ("classifier", KNeighborsClassifier())
])

params = {"classifier__n_neighbors": [3, 5, 10, 20], "classifier__p": [1, 2]}

grid_search = GridSearchCV(estimator=pipe, param_grid = params)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_
best_result = grid_search.best_score_

print(best_params)
print(f"Melhor score: {best_result:.4f}")

In [ ]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocess_svm = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ],
    remainder='passthrough'
)

pipe = Pipeline(steps = [
    ("preprocess", preprocess_svm),
    ("classifier", SVC())
])

params = {
    "classifier__tol": [0.001, 0.0001, 0.00001],
    "classifier__C": [1.0, 1.5, 2.0],
    "classifier__kernel": ["rbf", "linear", "poly", "sigmoid"]
}

grid_search = GridSearchCV(estimator=pipe, param_grid=params)
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(f"Melhor score: {grid_search.best_score_:.4f}")


In [ ]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocess_mlp = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ],
    remainder='passthrough'
)

pipe = Pipeline(steps = [
    ("preprocess", preprocess_mlp),
    ("classifier", MLPClassifier(max_iter=250))
])

params = {
    "classifier__batch_size": [10, 56],
    "classifier__solver": ["adam", "sgd"],
    "classifier__activation": ["relu", "logistic"]}

grid_search = GridSearchCV(estimator=pipe, param_grid = params)
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(f"Melhor score: {grid_search.best_score_:.4f}")

# Validação cruzada

In [14]:
result_tree = []
results_random_forest = []
results_kNN = []
results_svm = []
results_rede_neural = []

for i in range(5):
    kfold = KFold(n_splits=10, shuffle=True, random_state=i)

    # DECISION TREE
    pipe_tree = Pipeline([
        ("preprocess", preprocess),
        ("model", DecisionTreeClassifier(
            criterion="entropy",
            min_samples_leaf=10,
            min_samples_split=2,
            splitter="random"
        ))
    ])
    scores = cross_val_score(pipe_tree, X_train, y_train, cv=kfold)
    result_tree.append(scores.mean())

    # RANDOM FOREST
    pipe_rf = Pipeline([
        ("preprocess", preprocess),
        ("model", RandomForestClassifier(
            criterion="gini",
            min_samples_leaf=1,
            min_samples_split=10,
            n_estimators=40
        ))
    ])
    scores = cross_val_score(pipe_rf, X_train, y_train, cv=kfold)
    results_random_forest.append(scores.mean())

    # KNN
    pipe_knn = Pipeline([
        ("preprocess", preprocess),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", KNeighborsClassifier(
            n_neighbors=20,
            p=2
        ))
    ])
    scores = cross_val_score(pipe_knn, X_train, y_train, cv=kfold)
    results_kNN.append(scores.mean())

    # SVM
    pipe_svm = Pipeline([
        ("preprocess", preprocess),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", SVC(
            kernel="rbf",
            C=1.5,
            tol=0.001
        ))
    ])
    scores = cross_val_score(pipe_svm, X_train, y_train, cv=kfold)
    results_svm.append(scores.mean())

    # REDE NEURAL
    pipe_mlp = Pipeline([
        ("preprocess", preprocess),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifier(
            activation="relu",
            batch_size=56,
            solver="sgd",
            max_iter=100
        ))
    ])
    scores = cross_val_score(pipe_mlp, X_train, y_train, cv=kfold)
    results_rede_neural.append(scores.mean())


In [15]:
df_all_results = pd.DataFrame({
    "Decision Tree": result_tree,
    "Random Forest": results_random_forest,
    "KNN": results_kNN,
    "SVM": results_svm,
    "Rede Neural": results_rede_neural
})

df_all_results

,Decision Tree,Random Forest,KNN,SVM,Rede Neural
0,0.774045,0.787175,0.734463,0.712277,0.758064
1,0.778672,0.791271,0.734472,0.710509,0.754357
2,0.779196,0.792505,0.734458,0.706259,0.756117
3,0.777068,0.792506,0.734282,0.708023,0.751148
4,0.776714,0.792155,0.734465,0.707144,0.757365


# 📊 **Descrição dos Resultados dos Modelos**

Os experimentos foram realizados com 5 execuções de validação cruzada (K-Fold) para cada algoritmo, avaliando a acurácia média em cada repetição. Os resultados obtidos foram:  
| Modelo               | Acurácia Média Aproximada |
|----------------------|----------------------------|
| **Random Forest**    | **~0.792 ✅** |
| Rede Neural (MLP)    | ~0.755 |
| Decision Tree        | ~0.778 |
| KNN                  | ~0.734 |
| **SVM**              | **~0.709 ❌** |

#**Análise dos Resultados**

* 🔹 Random Forest foi o melhor modelo, apresentando a maior acurácia em todas as execuções, com resultados muito estáveis (~79%).

* 🔹 A Rede Neural (MLP) teve bom desempenho, ficando em segundo lugar (~75%), mostrando boa capacidade de aprendizado.

* 🔹 A Árvore de Decisão (Decision Tree) teve desempenho sólido (~78%), mas inferior ao Random Forest, o que já era esperado por ser um modelo isolado.

* 🔹 O KNN apresentou desempenho intermediário (~73%), com pouca variação entre as execuções.

* 🔹 O SVM foi o pior modelo nesse cenário, obtendo aproximadamente 71% de acurácia, indicando possível necessidade de ajuste de hiperparâmetros.

#**Conclusão Final**

O algoritmo Random Forest foi o modelo mais eficiente para este conjunto de |dados, apresentando a melhor taxa de acerto média e maior estabilidade, sendo o mais indicado para uso em produção ou como modelo final do projeto.

# 📈 **Benefícios dos Resultados**

##**1. Redução de Perda de Receita**

Com a previsão de quais clientes têm maior risco de cancelar, a empresa pode:

* Oferecer descontos personalizados

* Criar planos de retenção

* Melhorar o atendimento antes do cancelamento

📌 *Impacto direto*: menos cancelamentos = **mais faturamento recorrente**.

##**2. Ações de Retenção Mais Inteligentes**

Em vez de gastar dinheiro com campanhas para todos os clientes, a empresa passa a:

* Focar apenas nos clientes com maior risco

* Reduzir o custo de campanhas

* Ter maior taxa de sucesso nas ações comerciais

📌 *Impacto direto*: redução de custos operacionais.

##**3. Aumento da Eficiência Comercial**

A equipe de vendas passa a ter:

* Listas priorizadas de clientes críticos

* Argumentos baseados em dados

* Ofertas personalizadas por perfil de consumo

📌 *Impacto direto*: as decisões deixam de ser baseadas em “achismo”.

#**4. Melhoria na Experiência do Cliente**

Ao identificar padrões de churn, a empresa descobre:

* Se clientes cancelam por preço

* Por problemas de qualidade

* Por mau atendimento

* Por concorrência

📌 *Impacto direto*: melhora dos serviços e **aumento da satisfação do cliente.**

#**5. Planejamento Estratégico de Longo Prazo**

Com os modelos (como o Random Forest, que foi o melhor no seu caso), a empresa pode:

* Estimar o faturamento futuro

* Planejar expansão

* Criar novos produtos mais aderentes ao perfil dos clientes

📌 *Impacto direto*: decisões estratégicas mais seguras.

# **Conclusão Executiva**

A aplicação de modelos de Machine Learning para previsão de churn permite à empresa de telecom reduzir perdas financeiras, aumentar a eficiência comercial, melhorar a experiência do cliente e apoiar decisões estratégicas baseadas em dados, transformando dados operacionais em vantagem competitiva.